In [ ]:
## Importing Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

print("Libraries Imported.")

In [ ]:
## Loading Dataset

df = pd.read_csv(r"")
print("Data Loaded Successfully.")

In [ ]:
## Dataset Information

print("Shape: ", df.shape)

print("\n Coulmns: ")
print(df.columns)

print("\n Missing Values: ")
print(df.isnull().sum().sum())

In [ ]:
## Checking Depression Classes

print(df["Depression_Level"].value_counts())

print("\n Unique Classes: ")
print(sorted(df["Depression_Level"].unique()))

In [ ]:
## Selecting Features

X = df.drop(columns = ["Stress_Level", "Anxiety_Level", "Depression_Level"])
y = df["Depression_Level"]

print("Feature Shape: ", X.shape)
print("Target Shape: ", y.shape)

In [ ]:
## Encoding Labels

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print("Classes: ")
print(label_encoder.classes_)

In [ ]:
## Splitting Dataset

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, random_state = 42, stratify = y_temp)

print("Train: ", X_train.shape)
print("Validation: ", X_val.shape)
print("Test: ", X_test.shape)

In [ ]:
## Converting To Tensors

X_train = torch.tensor(X_train.values, dtype = torch.float32)
X_val = torch.tensor(X_val.values, dtype = torch.float32)
X_test = torch.tensor(X_test.values, dtype = torch.float32)

y_train = torch.tensor(y_train, dtype = torch.long)
y_val = torch.tensor(y_val, dtype = torch.long)
y_test = torch.tensor(y_test, dtype = torch.long)

print("Tensor Converion Complete")

In [ ]:
## Assigning Classes To Datasets

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class QuestionnaireDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return (self.features[idx], self.labels[idx])       

In [ ]:
## Creating Dataset

train_dataset = QuestionnaireDataset(X_train, y_train)
val_dataset = QuestionnaireDataset(X_val, y_val)
test_dataset = QuestionnaireDataset(X_test, y_test)

print("Datasets Created")

In [ ]:
## Creating DataLoaders

train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = True)

print("DataLoaders Created")

In [ ]:
## Verifying Shapes

features, labels = next(iter(train_loader))

print("Feature Shape: ", features.shape)
print("Label Shape: ", labels.shape)

In [ ]:
## Assigning Device

device = torch.device("cuda")
print("Device: ", device)

if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))

In [ ]:
## Neural Network

class QuestionnaireNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
                                        nn.Linear(30, 64),
                                        nn.ReLU(),
                                        nn.Dropout(0.3),

                                        nn.Linear(64, 32),
                                        nn.ReLU(),
                                        nn.Dropout(0.3),

                                        nn.Linear(32, 5)
                                    )
    
    def forward(self, x):
        return self.network(x)

In [ ]:
## Initilizing Model

model = QuestionnaireNN().to(device)
print(model)

In [ ]:
## Verifying Architecture

dummy_input = torch.randn(1, 30).to(device)
output = model(dummy_input)
print(output.shape)

In [ ]:
## Loss, Optimizer, Scheduler

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr = 0.001, weight_decay = 1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = "min", factor = 0.5, patience = 3)

print("Training Setup Complete.")

In [ ]:
## Creating Training Function

def training_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, labels in loader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100*correct/total
    
    return epoch_loss, epoch_acc

In [ ]:
## Creating Validation Function

def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for features, labels in loader:
            features = features.to(device)
            labels = labels.to(device)
            
            outputs = model(features)
            loss = criterion(outputs, labels)       
      
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100*correct/total
    
    return epoch_loss, epoch_acc

In [ ]:
## Creating Variables for Progress Check

epochs = 30
best_val_accuracy = 0.0

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

In [ ]:
## Training Loop

for epoch in range(epochs):
    train_loss, train_acc = training_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc

        torch.save(model.state_dict(), "Questionnaire_Model.pth")
        print("Model Saved.")

In [ ]:
## Loading Model

model.load_state_dict(torch.load("Questionnaire_Model.pth"))
model.eval()

print("Best Model Loaded")

In [ ]:
## Creating Evaluations

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():

    for features, labels in test_loader:

        features = features.to(device)

        outputs = model(features)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

print("Predictions Generated")

print("Labels:", len(all_labels))
print("Predictions:", len(all_preds))

In [ ]:
test_accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {test_accuracy*100:.2f} %")

In [ ]:
print(classification_report(all_labels, all_preds))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import torch

cm = confusion_matrix(all_labels, all_preds)

class_names = [
    "Normal",
    "Mild",
    "Moderate",
    "Severe",
    "Extremely Severe"
]

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title(
    "Questionnaire Model Confusion Matrix"
)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "True Label"
)

plt.tight_layout()

plt.show()